In [2]:
import re
import math
import time
import numpy as np
import torch
import torch.nn as nn
# import torch.profiler as profiler
# import torchvision
from torch.utils.data import Dataset, DataLoader
# import chess
from abc import ABC, abstractmethod
import ctypes
from ctypes import POINTER, c_int, c_float

In [3]:
SIZE = 7153281

BLACK_KING_BUCKETS = [0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1,
                      0, 0, 0, 0, 1, 1, 1, 1]

NUM_KING_BUCKETS = 2

# Step 1: Load the library
lib_path = "libamethyst_tuner.dylib"
libtuner = ctypes.cdll.LoadLibrary(lib_path)

# Step 2: Set the argtypes of all functions from the library
libtuner.getResults.argtypes = [POINTER(c_float), c_int]
libtuner.getPhases.argtypes = [POINTER(c_int), c_int]
libtuner.getKingSquares.argtypes = [c_int, POINTER(c_int), c_int]
libtuner.getMobility.argtypes = [c_int, POINTER(c_int), c_int]
libtuner.getPSTs.argtypes = [c_int, c_int, c_int, POINTER(c_int), c_int]

def get_results():
    results = (c_float * SIZE)()
    libtuner.getResults(results,
                       c_int(SIZE))
    results = np.ctypeslib.as_array(results, shape = (SIZE,))
    results = torch.tensor(results)
    return results

def get_phases():
    phases = (c_int * SIZE)()
    libtuner.getPhases(phases,
                       c_int(SIZE))
    phases = np.ctypeslib.as_array(phases, shape = (SIZE,))
    phases = torch.tensor(phases)
    phases = torch.stack([phases, 24 - phases])
    phases = torch.transpose(phases, 0, 1).float()
    return phases

def get_king_squares(side):
    kingSquares = (c_int * SIZE)()
    libtuner.getKingSquares(c_int(side), kingSquares, c_int(SIZE))
    kingSquares = np.ctypeslib.as_array(kingSquares, shape=(SIZE,))
    return kingSquares.reshape(-1,1)
    
def get_mobility(side):
    mobility = (c_int * (6 * SIZE))()
    libtuner.getMobility(c_int(side), mobility, c_int(SIZE))
    mobility = np.ctypeslib.as_array(mobility, shape=(6 * SIZE,))
    return mobility.reshape(-1,6)

def get_psts(side, piece):
    maxPieces = [8,2,2,2,2,1][piece]
    psts = (c_int * (maxPieces * SIZE))()
    libtuner.getPSTs(c_int(side),      # side
                     c_int(piece),     # piece
                     c_int(maxPieces), # maxPieces
                     psts,             # arr
                     c_int(SIZE))      # length
    
    psts = np.ctypeslib.as_array(psts, shape = (maxPieces * SIZE,))
    psts = psts.reshape(-1,maxPieces)
    psts.sort()
    return torch.tensor(psts)

def get_king_bucketed_psts(side, piece):
    psts = get_psts(side, piece)
    king_squares = get_king_squares(side).reshape(-1)
    cap = torch.tensor(64 * NUM_KING_BUCKETS)
    
    king_buckets = torch.tensor([BLACK_KING_BUCKETS[ksq ^ 56] for ksq in king_squares]).reshape(-1, 1)
    
    psts = torch.where(psts == 64, torch.tensor(10000), psts)
    psts += 64 * king_buckets
    psts = torch.minimum(psts, cap)
    return psts

In [4]:
def get_all_psts():
    return [torch.stack([get_psts(0,piece), get_psts(1, piece)]) for piece in range(6)]
def get_all_king_bucketed_psts():
    return [torch.stack([get_king_bucketed_psts(0,piece), get_king_bucketed_psts(1, piece)]) for piece in range(6)]

In [5]:
t0 = time.time()
ONE_HOT_TENSORS = get_all_king_bucketed_psts()
t1 = time.time()
print(f"Made embeddings in {t1-t0:.2f} seconds")

Made embeddings in 260.57 seconds


In [6]:
t0 = time.time()
mobility_diff = get_mobility(0) - get_mobility(1)
t1 = time.time()
print(f"Made mobility tensor in {t1-t0:.2f} seconds")

Made mobility tensor in 21.46 seconds


In [7]:
CONTINUOUS_TENSOR = torch.tensor(mobility_diff, dtype = torch.float32)

In [8]:
EMBEDDING_DIMS = [128, 128, 128, 128, 128, 128]
CONTINUOUS_FEATURE_NAMES = ["mobility"]
CONTINUOUS_FEATURE_DIMS = [6]

In [9]:
t0 = time.time()
RESULTS = get_results()
t1 = time.time()
print(f"Made results tensor in {t1-t0:.2f} seconds")

Made results tensor in 3.65 seconds


In [10]:
t0 = time.time()
PHASES = get_phases()
t1 = time.time()
print(f"Made phase tensor in {t1-t0:.2f} seconds")

Made phase tensor in 2.01 seconds


In [11]:
SCALING_TERM = 28 * 400 / 2.1 # Yes that should be a 24 instead of 28 and that's on my to do list to change

class Tuner(nn.Module):
    def __init__(self):
        super(Tuner, self).__init__()
        self.continuous_layer = nn.Linear(CONTINUOUS_TENSOR.shape[1], 2, bias = False)
        self.embedding_layers = [nn.Embedding(dim + 1, 2, padding_idx = dim) for dim in EMBEDDING_DIMS]
        with torch.no_grad():
            self.continuous_layer.weight.zero_()
            for i, layer in enumerate(self.embedding_layers):
                layer.weight.zero_()
                self.register_parameter(f"embedding{i}", layer.weight)
                
    def get_logits(self, layer, phases, embedding_tensor):
        embeddings = layer(embedding_tensor)
        summed = torch.sum(embeddings, axis = 2)
        summed2 = summed[0] - summed[1]
        return torch.sum(summed2 * phases, axis = 1)
        
        
    def forward(self, phases, continuous_tensor, embedding_tensors):
        assert len(embedding_tensors) == len(self.embedding_layers)
        logits = sum(self.get_logits(layer, phases, embedding_tensors[i]) for i, layer in enumerate(self.embedding_layers))
        logits += torch.sum(phases * self.continuous_layer(continuous_tensor), axis = 1)
        return torch.sigmoid(logits / SCALING_TERM)
    
    def forward_as_in_engine(self, phases, continuous_tensor, embedding_tensors):
        # This is the same as forward, but it divides by 24 instead of SCALING_TERM and it doesn't take the sigmoid
        logits = sum(self.get_logits(layer, phases, embedding_tensors[i]) for i, layer in enumerate(self.embedding_layers))
        logits += torch.sum(phases * self.continuous_layer(continuous_tensor), axis = 1)
        # We don't need to worry about setting these to ints. The other thing will do it for us.
        return logits / 24
    
    def round_weights(self):
        with torch.no_grad():
            self.continuous_layer.weight.round_()
            for layer in self.embedding_layers:
                layer.weight.round_()


In [12]:
def get_detached_params(model):
    return [param.detach().clone() for param in model.parameters()]

def get_max_delta(params1, params2):
    overall_max_delta = 0
    for param1, param2 in zip(params1, params2):
        deltas = torch.abs(param2 - param1)
        max_delta = torch.max(deltas)
        overall_max_delta = max(max_delta, overall_max_delta)
        
    return overall_max_delta

In [13]:
def get_model(model = None, optimizer = "lbfgs", max_epochs = 3200):
    # 1. Random stuff we need
    if model is None:
        model = Tuner()
    t0 = time.time()
    optimizer_str = optimizer
    
    # 2. The actual stuff needed for training
    loss_function = nn.BCELoss()
    if optimizer_str == "lbfgs":
        optimizer = torch.optim.LBFGS(model.parameters())
    elif optimizer_str == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr = 10)
    else:
        assert False # Optimizer must be "lbfgs" or "adam"
    
    def closure():
        # This is the closure required for LBFGS
        optimizer.zero_grad()
        y_pred = model(PHASES, CONTINUOUS_TENSOR, ONE_HOT_TENSORS)
        loss = loss_function(y_pred, RESULTS)
        loss.backward()
        return loss
    
    # 3. Stopping criteria
    STOPPING_DELTA = 0.5
    previous_params = None
    current_params = None
    
    # 4. Start the training loop
    for epoch in range(max_epochs):
        # 4A. The actual important training
        y_pred = model(PHASES, CONTINUOUS_TENSOR, ONE_HOT_TENSORS)
        loss = loss_function(y_pred, RESULTS)
        loss.backward()
        if optimizer_str == "lbfgs":
            optimizer.step(closure)
        else:
            optimizer.step()
        optimizer.zero_grad()

        # 4B. Printouts
        t1 = time.time()
        if True:
            print(f"epoch {epoch+1}; loss = {loss.item():.4f}; total time = {int(t1-t0)}")
            
        # 4C. Stopping criteria
        if (epoch % 10 == 9):
            current_params = get_detached_params(model)
            if previous_params is not None:
                max_delta = get_max_delta(previous_params, current_params)
                print(f"Max parameter weight change is {max_delta:.4f}")
                if max_delta < STOPPING_DELTA:
                    break

            previous_params = current_params

    model.round_weights()
    return model

In [14]:
MATERIAL_START_WEIGHTS = torch.tensor([[ 102,  394,  432,  499, 1103,    0],
                                       [ 196,  511,  538,  919, 1628,    0]])

In [15]:
model1 = Tuner()
with torch.no_grad():
    for piece in range(6):
        for phase in range(2):
            model1.embedding_layers[piece].weight[:,phase] = MATERIAL_START_WEIGHTS[phase,piece]
            # IMPORTANT: We need to set the weights for the padding index to 0
            model1.embedding_layers[piece].weight[NUM_KING_BUCKETS * 64] = 0

In [ ]:
model2 = get_model(model1, optimizer = "lbfgs", max_epochs = 100)

In [ ]:
model3 = get_model(model1, optimizer = "adam")

epoch 1; loss = 0.4001; total time = 6
epoch 2; loss = 0.4212; total time = 12
epoch 3; loss = 0.4025; total time = 18
epoch 4; loss = 0.4064; total time = 24
epoch 5; loss = 0.4119; total time = 30
epoch 6; loss = 0.4083; total time = 36
epoch 7; loss = 0.4026; total time = 42
epoch 8; loss = 0.4012; total time = 48
epoch 9; loss = 0.4040; total time = 55
epoch 10; loss = 0.4059; total time = 61
epoch 11; loss = 0.4046; total time = 67
epoch 12; loss = 0.4021; total time = 73
epoch 13; loss = 0.4009; total time = 80
epoch 14; loss = 0.4015; total time = 86
epoch 15; loss = 0.4027; total time = 92
epoch 16; loss = 0.4030; total time = 99
epoch 17; loss = 0.4020; total time = 105
epoch 18; loss = 0.4009; total time = 112
epoch 19; loss = 0.4006; total time = 119
epoch 20; loss = 0.4012; total time = 126
Max parameter weight change is 24.9419
epoch 21; loss = 0.4017; total time = 132
epoch 22; loss = 0.4015; total time = 139
epoch 23; loss = 0.4008; total time = 145
epoch 24; loss = 0.40

epoch 180; loss = 0.4000; total time = 1638
Max parameter weight change is 9.4451
epoch 181; loss = 0.4000; total time = 1644
epoch 182; loss = 0.4000; total time = 1650
epoch 183; loss = 0.4000; total time = 1656
epoch 184; loss = 0.4000; total time = 1662
epoch 185; loss = 0.4000; total time = 1668
epoch 186; loss = 0.4000; total time = 1673
epoch 187; loss = 0.4000; total time = 1679
epoch 188; loss = 0.4000; total time = 1685
epoch 189; loss = 0.4000; total time = 1691
epoch 190; loss = 0.4000; total time = 1697
Max parameter weight change is 8.9895
epoch 191; loss = 0.4000; total time = 1703
epoch 192; loss = 0.4000; total time = 1709
epoch 193; loss = 0.4000; total time = 1715
epoch 194; loss = 0.4000; total time = 1721
epoch 195; loss = 0.4000; total time = 1727
epoch 196; loss = 0.4000; total time = 1733
epoch 197; loss = 0.4000; total time = 1738
epoch 198; loss = 0.4000; total time = 1744
epoch 199; loss = 0.4000; total time = 1750
epoch 200; loss = 0.4000; total time = 1756


epoch 351; loss = 0.3999; total time = 4125
epoch 352; loss = 0.3999; total time = 4132
epoch 353; loss = 0.3999; total time = 4139
epoch 354; loss = 0.3999; total time = 4145
epoch 355; loss = 0.3999; total time = 4152
epoch 356; loss = 0.3999; total time = 4159
epoch 357; loss = 0.3999; total time = 4166
epoch 358; loss = 0.3999; total time = 5158
epoch 359; loss = 0.3999; total time = 5164
epoch 360; loss = 0.3999; total time = 5170
Max parameter weight change is 4.8137
epoch 361; loss = 0.3999; total time = 5176
epoch 362; loss = 0.3999; total time = 5182
epoch 363; loss = 0.3999; total time = 5189
epoch 364; loss = 0.3999; total time = 5195
epoch 365; loss = 0.3999; total time = 5201
epoch 366; loss = 0.3999; total time = 5208
epoch 367; loss = 0.3999; total time = 5215
epoch 368; loss = 0.3999; total time = 5222
epoch 369; loss = 0.3999; total time = 5228
epoch 370; loss = 0.3999; total time = 5235
Max parameter weight change is 4.7122
epoch 371; loss = 0.3999; total time = 5550


epoch 523; loss = 0.3999; total time = 10361
epoch 524; loss = 0.3999; total time = 10367
epoch 525; loss = 0.3999; total time = 10374
epoch 526; loss = 0.3999; total time = 10380
epoch 527; loss = 0.3999; total time = 10387
epoch 528; loss = 0.3999; total time = 10394
epoch 529; loss = 0.3999; total time = 10401
epoch 530; loss = 0.3999; total time = 10407
Max parameter weight change is 3.5157
epoch 531; loss = 0.3999; total time = 10414
epoch 532; loss = 0.3999; total time = 10421
epoch 533; loss = 0.3999; total time = 10427
epoch 534; loss = 0.3999; total time = 10434
epoch 535; loss = 0.3999; total time = 10441
epoch 536; loss = 0.3999; total time = 10448
epoch 537; loss = 0.3999; total time = 10455
epoch 538; loss = 0.3999; total time = 10462
epoch 539; loss = 0.3999; total time = 10468
epoch 540; loss = 0.3999; total time = 10475
Max parameter weight change is 3.4440
epoch 541; loss = 0.3999; total time = 10482
epoch 542; loss = 0.3999; total time = 10489
epoch 543; loss = 0.3999

epoch 691; loss = 0.3999; total time = 16555
epoch 692; loss = 0.3999; total time = 16561
epoch 693; loss = 0.3999; total time = 16567
epoch 694; loss = 0.3999; total time = 16574
epoch 695; loss = 0.3999; total time = 16581
epoch 696; loss = 0.3999; total time = 16587
epoch 697; loss = 0.3999; total time = 16594
epoch 698; loss = 0.3999; total time = 16601
epoch 699; loss = 0.3999; total time = 16607
epoch 700; loss = 0.3999; total time = 17554
Max parameter weight change is 2.3308
epoch 701; loss = 0.3999; total time = 17560
epoch 702; loss = 0.3999; total time = 17566
epoch 703; loss = 0.3999; total time = 17572
epoch 704; loss = 0.3999; total time = 17579
epoch 705; loss = 0.3999; total time = 17585
epoch 706; loss = 0.3999; total time = 17591
epoch 707; loss = 0.3999; total time = 17597
epoch 708; loss = 0.3999; total time = 17603
epoch 709; loss = 0.3999; total time = 17610
epoch 710; loss = 0.3999; total time = 17617
Max parameter weight change is 2.2672
epoch 711; loss = 0.3999

epoch 860; loss = 0.3999; total time = 25701
Max parameter weight change is 1.3586
epoch 861; loss = 0.3999; total time = 25707
epoch 862; loss = 0.3999; total time = 25713
epoch 863; loss = 0.3999; total time = 25721
epoch 864; loss = 0.3999; total time = 25729
epoch 865; loss = 0.3999; total time = 25735
epoch 866; loss = 0.3999; total time = 26684
epoch 867; loss = 0.3999; total time = 26690
epoch 868; loss = 0.3999; total time = 26697
epoch 869; loss = 0.3999; total time = 26703
epoch 870; loss = 0.3999; total time = 26709
Max parameter weight change is 1.3118
epoch 871; loss = 0.3999; total time = 26715
epoch 872; loss = 0.3999; total time = 26721
epoch 873; loss = 0.3999; total time = 26727
epoch 874; loss = 0.3999; total time = 26734
epoch 875; loss = 0.3999; total time = 26739
epoch 876; loss = 0.3999; total time = 26745
epoch 877; loss = 0.3999; total time = 26751
epoch 878; loss = 0.3999; total time = 26757
epoch 879; loss = 0.3999; total time = 26763
epoch 880; loss = 0.3999

epoch 1028; loss = 0.4000; total time = 27644
epoch 1029; loss = 0.4000; total time = 27650
epoch 1030; loss = 0.4001; total time = 27656
Max parameter weight change is 2.3059
epoch 1031; loss = 0.4001; total time = 27662
epoch 1032; loss = 0.4001; total time = 27668
epoch 1033; loss = 0.4000; total time = 27674
epoch 1034; loss = 0.3999; total time = 27680
epoch 1035; loss = 0.3999; total time = 27686
epoch 1036; loss = 0.4000; total time = 27692
epoch 1037; loss = 0.4000; total time = 27698
epoch 1038; loss = 0.4000; total time = 27704
epoch 1039; loss = 0.3999; total time = 27710
epoch 1040; loss = 0.3999; total time = 27716
Max parameter weight change is 1.7393
epoch 1041; loss = 0.3999; total time = 27722
epoch 1042; loss = 0.4000; total time = 27728
epoch 1043; loss = 0.4000; total time = 27734
epoch 1044; loss = 0.3999; total time = 27740
epoch 1045; loss = 0.3999; total time = 27746
epoch 1046; loss = 0.3999; total time = 27752
epoch 1047; loss = 0.3999; total time = 27758
epoc

In [ ]:
tuner4 = get_model(optimizer = "lbfgs")

epoch 1; loss = 0.6931; total time = 149
epoch 2; loss = 0.5257; total time = 300
epoch 3; loss = 0.4903; total time = 457
epoch 4; loss = 0.4608; total time = 616
epoch 5; loss = 0.4326; total time = 779
epoch 6; loss = 0.4169; total time = 937
epoch 7; loss = 0.4129; total time = 1089
epoch 8; loss = 0.4108; total time = 1222
epoch 9; loss = 0.4088; total time = 1354
epoch 10; loss = 0.4066; total time = 1486
epoch 11; loss = 0.4045; total time = 1618
epoch 12; loss = 0.4036; total time = 1750
epoch 13; loss = 0.4031; total time = 1882
epoch 14; loss = 0.4026; total time = 2013
epoch 15; loss = 0.4022; total time = 2145
epoch 16; loss = 0.4019; total time = 2276
epoch 17; loss = 0.4017; total time = 2408
epoch 18; loss = 0.4016; total time = 2539
epoch 19; loss = 0.4014; total time = 2671
epoch 20; loss = 0.4013; total time = 2803
Max parameter weight change is 610.0314
epoch 21; loss = 0.4012; total time = 2935
epoch 22; loss = 0.4011; total time = 3067
epoch 23; loss = 0.4011; tota

In [ ]:
def print_psts(params):
    print("constexpr std::array<std::array<packed_eval_t, 64>, 6> psts = {{")
    for piece_type in range(6):
        print("{")

        # Determine how much whitespace we need
        l1 = len(str(torch.min(params[piece_type].weight.data.int()).item()))
        l2 = len(str(torch.max(params[piece_type].weight.data.int()).item()))
        l = max(l1, l2)

        for i, thing in enumerate(params[piece_type].weight.data.int()[0:64,:]):
            print(f"S({thing[0]:{l}},{thing[1]:{l}}), ", end = "\n" * (i % 8 - 6))
        print("},")

    print("}};")

In [ ]:
def print_king_bucketed_psts(params):
    print(f"constexpr std::array<std::array<std::array<packed_eval_t, 64>, 6>, {NUM_KING_BUCKETS}> psts = " + "{{")
    for bucket in range(NUM_KING_BUCKETS):
        idx = bucket * 64
        print("{{")
        for piece_type in range(6):
            print("{")

            # Determine how much whitespace we need
            l1 = len(str(torch.min(params[piece_type].weight.data.int()).item()))
            l2 = len(str(torch.max(params[piece_type].weight.data.int()).item()))
            l = max(l1, l2)

            for i, thing in enumerate(params[piece_type].weight.data.int()[idx:idx+64,:]):
                print(f"S({thing[0]:{l}},{thing[1]:{l}}), ", end = "\n" * (i % 8 - 6))
            print("},")
        print("}},")
    print("}};")

In [ ]:
eval_t_name = "packed_eval_t"
def print_continuous_weights(model):
    printout = "";
    # Step 1: Continuous weights
    weights_tensor = model.continuous_layer.weight
    index = 0
    for feature_name, dims in zip(CONTINUOUS_FEATURE_NAMES, CONTINUOUS_FEATURE_DIMS):
#         feature_name = feature.get_name()
#         dims = feature.get_n()
        feature_weights = torch.round(weights_tensor[:, index:index+dims])
        str_feature_weights = [f"S{int(feature_weights[0, i]), int(feature_weights[1, i])}" for i in range(feature_weights.shape[1])]
        strw2 = "{" + ", ".join(str_feature_weights) + "}"
        line = f"constexpr {eval_t_name} {feature_name}[{dims}] = {strw2};"
        print(line)
        index += dims

In [ ]:
print_continuous_weights(model2)

constexpr packed_eval_t mobility[6] = {S(-13, -26), S(0, 1), S(9, 12), S(8, 6), S(3, 9), S(-17, -3)};


In [ ]:
print_king_bucketed_psts(model2.embedding_layers)

constexpr std::array<std::array<std::array<packed_eval_t, 64>, 6>, 2> psts = {{
{{
{
S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), 
S(156,249), S(240,262), S(195,246), S(130,270), S( 91,283), S( 98,279), S( 65,306), S(  0,289), 
S(166,238), S(195,261), S(185,237), S(119,270), S(132,253), S( 97,257), S( 63,274), S( -2,262), 
S(144,272), S(190,288), S(180,253), S(163,263), S(150,253), S(118,255), S(100,278), S( 13,260), 
S(162,335), S(204,343), S(186,304), S(144,295), S(173,271), S(151,264), S(114,303), S( 34,295), 
S(167,498), S(193,551), S(164,504), S(190,435), S(197,414), S(203,359), S(134,431), S( 34,410), 
S(202,627), S(484,508), S(270,664), S(302,553), S(185,560), S(173,502), S( 88,585), S(-31,594), 
S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), S(102,196), 
},
{
S(308,551), S(345,583), S(366,597), S(380,600), S(373,579), S(376,571), S(374,563), S(321,506), 
S(375,572), S(337,624), S(412,617), S

In [ ]:
def save_eval_to_file(model, filename):
    evals = model.forward_as_in_engine(PHASES, CONTINUOUS_TENSOR, ONE_HOT_TENSORS)
    with open(filename, "w") as file:
        for eval_ in evals:
            file.write(str(int(eval_.item())))
            file.write('\n')
            
    print(f"len(evals) is {len(evals)}")
    file.close()

In [ ]:
save_eval_to_file(model2,"NoahResults.txt")

len(evals) is 7153281


In [ ]:
def save_model(model, filename):
    torch.save(model, "models/" + filename + ".pth") 
    # Load this with: model = torch.load(filename)
    # maybe do: model.eval()
    
    torch.save(model.state_dict(), "models/" + filename + "_state_dict.pth")
    # Load this with:
    # model = Tuner()
    # model.load_state_dict(torch.load(filename))
    # maybe do: model.eval()
    

In [ ]:
save_model(model2, "amethyst_196_hce")

In [ ]:
print_continuous_weights(tuner4)

constexpr packed_eval_t mobility[6] = {S(-13, -25), S(0, 2), S(9, 12), S(8, 6), S(2, 12), S(-17, -3)};


In [ ]:
print_king_bucketed_psts(tuner4.embedding_layers)

constexpr std::array<std::array<std::array<packed_eval_t, 64>, 6>, 2> psts = {{
{{
{
S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), 
S(161,248), S(246,261), S(199,244), S(133,270), S( 94,281), S(101,278), S( 68,304), S(  2,289), 
S(171,236), S(200,260), S(189,236), S(121,270), S(136,251), S( 99,255), S( 66,273), S(  1,261), 
S(150,270), S(196,286), S(184,252), S(165,262), S(153,252), S(120,254), S(103,277), S( 14,259), 
S(167,333), S(209,342), S(188,303), S(147,295), S(175,270), S(154,263), S(119,301), S( 35,295), 
S(185,495), S(207,550), S(176,500), S(200,434), S(209,412), S(206,360), S(139,432), S( 38,410), 
S(230,622), S(500,505), S(274,666), S(305,546), S(195,567), S(176,499), S( 99,588), S(-20,594), 
S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), S(  0,  0), 
},
{
S(325,552), S(366,579), S(386,597), S(397,602), S(389,582), S(396,571), S(391,565), S(319,499), 
S(398,569), S(360,620), S(440,611), S

In [ ]:
save_eval_to_file(tuner4, "NoahResults.txt")

len(evals) is 7153281


In [ ]:
save_model(tuner4, "amethyst_197_hce")